1.Setup Imports & Helper Functions

In [ ]:
import json
import re
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine

# Helper Functions

def clean_nric(ic_str):
    """
    Clean fictional customer identification values.

    Expected example: MOCK-IC-0001-1989
    Missing or invalid identifiers become None.
    """
    if pd.isna(ic_str) or not str(ic_str).strip():
        return None

    ic_str = str(ic_str).strip().upper()

    if re.fullmatch(r"MOCK-IC-\d{4}-\d{4}", ic_str):
        return ic_str

    return None




def clean_phone(phone_str):
    """
    Clean fictional phone identifiers.

    Expected example: MOCK-PHONE-0001
    Missing or invalid values become None.
    """
    if pd.isna(phone_str) or not str(phone_str).strip():
        return None

    phone_str = str(phone_str).strip().upper()

    if re.fullmatch(r"MOCK-PHONE-\d{4}", phone_str):
        return phone_str

    return None


def clean_gender(gender_str):
    """
    Standardize gender values to M or F.
    Missing or unexpected values become None.
    """
    if pd.isna(gender_str) or not str(gender_str).strip():
        return None

    gender = str(gender_str).strip().upper()

    gender_mapping = {
        "M": "M",
        "MALE": "M",
        "F": "F",
        "FEMALE": "F"
    }

    return gender_mapping.get(gender)


def clean_account_type(account_type_str):
    """
    Standardize account types into consistent categories.
    Invalid or missing account types become None.
    """
    if pd.isna(account_type_str) or not str(account_type_str).strip():
        return None

    account_type = str(account_type_str).strip().upper()

    account_type_mapping = {
        "SAVINGS": "SAVINGS",
        "SAVINGS_ACC": "SAVINGS",
        "CURRENT": "CURRENT",
        "CURRENT_ACC": "CURRENT",
        "FIXED DEPOSIT": "FIXED_DEPOSIT",
        "FIXED_DEPOSIT": "FIXED_DEPOSIT"
    }

    return account_type_mapping.get(account_type)


def clean_account_status(status_str):
    """
    Standardize account status values.
    Missing or unexpected values become None.
    """
    if pd.isna(status_str) or not str(status_str).strip():
        return None

    status = str(status_str).strip().upper()

    status_mapping = {
        "ACTIVE": "ACTIVE",
        "A": "ACTIVE",
        "DORMANT": "DORMANT",
        "CLOSED": "CLOSED"
    }

    return status_mapping.get(status)



def clean_transaction_type(txn_type_str):
    """
    Standardize transaction types into consistent categories.
    Missing or invalid values become None.
    """
    if pd.isna(txn_type_str) or not str(txn_type_str).strip():
        return None

    txn_type = str(txn_type_str).strip().upper()

    transaction_mapping = {
        "DEPOSIT": "DEPOSIT",
        "DEP": "DEPOSIT",
        "WITHDRAWAL": "WITHDRAWAL",
        "WD": "WITHDRAWAL",
        "TRANSFER": "TRANSFER"
    }

    return transaction_mapping.get(txn_type)


def clean_transaction_datetime(date_str):
    """
    Standardize transaction timestamps.

    Supported formats:
    - YYYY-MM-DD HH:MM:SS
    - DD/MM/YYYY HH:MM

    Invalid or missing values become None.
    """
    if pd.isna(date_str) or not str(date_str).strip():
        return None

    date_str = str(date_str).strip()

    supported_formats = [
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y %H:%M"
    ]

    for fmt in supported_formats:
        try:
            parsed_date = datetime.strptime(date_str, fmt)
            return parsed_date.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            continue

    return None


def clean_date(date_str):
  if pd.isna(date_str):
    return None
  date_str = str(date_str).strip()
  for fmt in [
      "%Y-%m-%d",
      "%d/%m/%Y",
      "%d-%m-%Y",
      "%Y/%m/%d",
      "%Y-%m-%d %H:%M:%S",
  ]:
    try:
      return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
    except ValueError:
      continue
  return None


print("✓ Helpers loaded successfully")

2.Process Customers

In [ ]:
# Load Customer Mapping & Data
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "mappings").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

with open(PROJECT_ROOT / "mappings" / "customer_mapping.json", encoding="utf-8") as f:
    cust_map = json.load(f)

cust_df = pd.read_csv(PROJECT_ROOT / cust_map["source_entity"])

# Clean & Transform
cust_df = cust_df.drop_duplicates(subset=["cust_id"], keep="first")
cust_df["full_name"] = cust_df["full_name"].str.title().str.strip()
cust_df["ic_no"] = cust_df["ic_no"].apply(clean_nric)
cust_df["phone_no"] = cust_df["phone_no"].apply(clean_phone)
cust_df["dob"] = cust_df["dob"].apply(clean_date)
cust_df["created_at"] = cust_df["created_at"].apply(clean_date)
cust_df["gender"] = cust_df["gender"].apply(clean_gender)

# Rename columns
cust_renames = {
    k: v["target_column"] for k, v in cust_map["column_mappings"].items()
}
cust_df = cust_df.rename(columns=cust_renames)


# ==========================================
# CUSTOMER DATA VALIDATION
# ==========================================

# Required fields based on customer_mapping.json
required_customer_columns = [
    "customer_id",
    "full_name",
    "created_date"
]

# Count records before validation
customers_before = len(cust_df)

# Treat whitespace-only required fields as missing
for column in required_customer_columns:
    cust_df[column] = cust_df[column].replace(r"^\s*$", pd.NA, regex=True)

# Remove records missing required fields
cust_df = cust_df.dropna(
    subset=required_customer_columns
).copy()

# Standardize customer IDs
cust_df["customer_id"] = (
    cust_df["customer_id"].astype(str).str.strip().str.upper()
)

# Count rejected records
customers_rejected = customers_before - len(cust_df)

print("CUSTOMER VALIDATION")
print(f"Records before validation: {customers_before}")
print(f"Records rejected: {customers_rejected}")
print(f"Records remaining: {len(cust_df)}")


# Inspect cleaned data
cust_df.head()

3.Process Accounts

In [ ]:
with open(
    PROJECT_ROOT / "mappings" / "account_mapping.json",
    encoding="utf-8"
) as f:
    acc_map = json.load(f)

acc_df = pd.read_csv(PROJECT_ROOT / acc_map["source_entity"])

# Keep valid customer references only (Orphan Check)
valid_cust_ids = set(cust_df["customer_id"])
acc_df = acc_df[acc_df["cust_id"].isin(valid_cust_ids)].copy()

acc_df["opened_date"] = acc_df["opened_date"].apply(clean_date)
acc_df["account_type"] = acc_df["account_type"].apply(clean_account_type)
acc_df["status"] = acc_df["status"].apply(clean_account_status)
# Clean balance directly in the existing 'balance' column
acc_df["balance"] = acc_df["balance"].apply(
    lambda x: abs(x) if pd.notna(x) else 0.0
)

# Rename columns (this safely turns 'balance' into 'current_balance')
acc_renames = {
    k: v["target_column"] for k, v in acc_map["column_mappings"].items()
}
acc_df = acc_df.rename(columns=acc_renames)


# ==========================================
# ACCOUNT DATA VALIDATION
# ==========================================

# Required fields based on account_mapping.json
required_account_columns = [
    "account_number",
    "customer_id",
    "account_type",
    "current_balance",
    "account_status",
    "opened_date"
]

accounts_before = len(acc_df)

# Treat whitespace-only values as missing
for column in required_account_columns:
    acc_df[column] = acc_df[column].replace(
        r"^\s*$", pd.NA, regex=True
    )

# Remove records with missing required fields
acc_df = acc_df.dropna(
    subset=required_account_columns
).copy()

# Standardize primary and foreign key values
acc_df["account_number"] = (
    acc_df["account_number"].astype(str).str.strip().str.upper()
)

acc_df["customer_id"] = (
    acc_df["customer_id"].astype(str).str.strip().str.upper()
)

# Remove duplicate account numbers
acc_df = acc_df.drop_duplicates(
    subset=["account_number"],
    keep="first"
).copy()

# Ensure each account belongs to an existing customer
acc_df = acc_df[
    acc_df["customer_id"].isin(cust_df["customer_id"])
].copy()

accounts_rejected = accounts_before - len(acc_df)

print("ACCOUNT VALIDATION")
print(f"Records before validation: {accounts_before}")
print(f"Records rejected: {accounts_rejected}")
print(f"Records remaining: {len(acc_df)}")


# Inspect cleaned data
acc_df.head()

4.Process Transaction

In [ ]:
import json
import pandas as pd

# 1. Load Transaction Mapping & Data
with open(
    PROJECT_ROOT / "mappings" / "transaction_mapping.json",
    encoding="utf-8"
) as f:
    txn_map = json.load(f)

txn_df = pd.read_csv(PROJECT_ROOT / txn_map["source_entity"])

# 2. Rename columns FIRST so all transformations match the expected target schema
txn_renames = {
    k: v["target_column"] for k, v in txn_map["column_mappings"].items()
}
txn_df = txn_df.rename(columns=txn_renames)

# Handle schema column naming alignment if JSON maps differently than DDL table
column_fixes = {
    "transaction_amount": "amount",
    "transaction_timestamp": "transaction_date",
    "txn_date": "transaction_date",
    "account_id": "account_number",
}
txn_df = txn_df.rename(
    columns={k: v for k, v in column_fixes.items() if k in txn_df.columns}
)

# 3. Foreign Key Check (Keep valid account references only)
valid_acc_numbers = set(acc_df["account_number"])
txn_df = txn_df[txn_df["account_number"].isin(valid_acc_numbers)].copy()

# 4. Clean & Transform Fields
txn_df["transaction_date"] = txn_df["transaction_date"].apply(
    clean_transaction_datetime
)
txn_df["amount"] = txn_df["amount"].apply(
    lambda x: abs(x) if pd.notna(x) else 0.0
)

txn_df["transaction_type"] = txn_df["transaction_type"].apply(
    clean_transaction_type
)


# ==========================================
# TRANSACTION DATA VALIDATION
# ==========================================

# Required fields based on transaction_mapping.json
# These use the column names from the current ETL notebook.
required_transaction_columns = [
    "transaction_id",
    "account_number",
    "transaction_type",
    "amount",
    "transaction_date"
]

transactions_before = len(txn_df)

# Treat whitespace-only values as missing
for column in required_transaction_columns:
    txn_df[column] = txn_df[column].replace(
        r"^\s*$", pd.NA, regex=True
    )

# Remove transactions with missing required fields
txn_df = txn_df.dropna(
    subset=required_transaction_columns
).copy()

# Standardize transaction and account identifiers
txn_df["transaction_id"] = (
    txn_df["transaction_id"].astype(str).str.strip().str.upper()
)

txn_df["account_number"] = (
    txn_df["account_number"].astype(str).str.strip().str.upper()
)

# Remove duplicate transactions
txn_df = txn_df.drop_duplicates(
    subset=["transaction_id"],
    keep="first"
).copy()

# Ensure transactions belong to valid accounts
txn_df = txn_df[
    txn_df["account_number"].isin(acc_df["account_number"])
].copy()

transactions_rejected = transactions_before - len(txn_df)

print("TRANSACTION VALIDATION")
print(f"Records before validation: {transactions_before}")
print(f"Records rejected: {transactions_rejected}")
print(f"Records remaining: {len(txn_df)}")


# Inspect cleaned data
txn_df.head()

4.Test Database Connection & Push Data

In [ ]:

import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ==========================================
# OPTIONAL POSTGRESQL LOADING
# ==========================================

# False = Clean and validate data without PostgreSQL
# True  = Connect to PostgreSQL and load the tables
LOAD_TO_POSTGRES = False

if LOAD_TO_POSTGRES:

    # Read database settings from environment variables
    DB_USER = os.getenv("DB_USER", "postgres")
    DB_PASS = os.environ["DB_PASSWORD"]
    DB_HOST = os.getenv("DB_HOST", "localhost")
    DB_PORT = os.getenv("DB_PORT", "5432")
    DB_NAME = os.getenv("DB_NAME", "banking_learning_db")

    # Build connection securely
    connection_url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )

    engine = create_engine(connection_url)

    try:
        # Test database connection
        with engine.connect() as conn:
            print("Connected to PostgreSQL successfully.")

        # Load parent table first, then child tables
        cust_df.to_sql(
            "customers",
            engine,
            if_exists="append",
            index=False,
            chunksize=500
        )
        print("Customers loaded successfully.")

        acc_df.to_sql(
            "accounts",
            engine,
            if_exists="append",
            index=False,
            chunksize=500
        )
        print("Accounts loaded successfully.")

        txn_df.to_sql(
            "transactions",
            engine,
            if_exists="append",
            index=False,
            chunksize=500
        )
        print("Transactions loaded successfully.")

    except Exception as error:
        print(f"PostgreSQL loading failed: {error}")

    finally:
        engine.dispose()

else:
    print("PostgreSQL loading skipped.")
    print("Cleaned datasets are available in memory:")
    print(f"Customers: {len(cust_df)} records")
    print(f"Accounts: {len(acc_df)} records")
    print(f"Transactions: {len(txn_df)} records")


In [ ]:

# ==========================================
# EXPORT CLEANED DATASETS
# ==========================================

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cust_df.to_csv(OUTPUT_DIR / "customers_clean.csv", index=False)
acc_df.to_csv(OUTPUT_DIR / "accounts_clean.csv", index=False)
txn_df.to_csv(OUTPUT_DIR / "transactions_clean.csv", index=False)

print("Cleaned datasets exported successfully.")
print(f"Output folder: {OUTPUT_DIR}")
